![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)

# Use watsonx to generate advertising


#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.

## Notebook content

This notebook contains the steps and code to demonstrate support of text generation in watsonx. It introduces commands for data generation and model testing.

Some familiarity with Python is helpful. This notebook uses Python 3.12.

## Learning goal

The goal of this notebook is to demonstrate how to use `watsonx.ai` model to generate mail advertising

## Contents

This notebook contains the following parts:

- [Setup](#setup)
- [Data loading](#data)
- [Foundation Models on watsonx](#models)
- [Model testing](#predict)
- [Summary](#summary)


<a id="setup"></a>

## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

- Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).


### Install and import the `datasets` and dependecies


In [ ]:
%pip install wget | tail -n 1
%pip install httpx | tail -n 1
%pip install ibm-cloud-sdk-core | tail -n 1

In [ ]:
import getpass
import os

import httpx
from ibm_cloud_sdk_core import IAMTokenManager
from IPython.display import Markdown, display

### Inferencing class

This cell defines a class that makes a REST API call to the watsonx Foundation Model
inferencing API that we will use to generate output from the provided input.
The class takes the access token created in the previous step, and uses it to
make a REST API call with input, model id and model parameters. The response
from the API call is returned as the cell output.


**Action:** Provide watsonx.ai Runtime url to work with watsonx.ai.


In [3]:
endpoint_url = input("Please enter your watsonx.ai Runtime endpoint url (hit enter): ")

Define a `Prompt` class for prompts generation.


In [4]:
class PromptClient:
    def __init__(self, access_token: str, project_id: str, endpoint_url: str):
        self.project_id = project_id
        self.url = f"{endpoint_url.rstrip('/')}/ml/v1/text/chat"
        self.headers = {
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        }

    def chat(self, model_id: str, messages: list[str], **params):
        payload = {
            "model_id": model_id,
            "messages": messages,
            "project_id": self.project_id,
            **params,
        }

        response = httpx.post(
            self.url,
            params={"version": "2024-03-19"},
            json=payload,
            headers=self.headers,
            timeout=30,
        )
        if response.status_code == 200:
            return response.json()
        else:
            raise RuntimeError(response.text)

### watsonx API connection

This cell defines the credentials required to work with watsonx API for Foundation
Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see <a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank" rel="IBM Cloud user API key">documentation</a>.


In [5]:
access_token = IAMTokenManager(
    apikey=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
    url="https://iam.cloud.ibm.com/identity/token",
).get_token()

### Defining the project id

The API requires project id that provides the context for the call. We will obtain
the id from the project in which this notebook runs. Otherwise, please provide the project id.


In [6]:
try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = getpass.getpass("Please enter your project_id (hit enter): ")

<a id="data"></a>

## Advertising prompts


#### Prepare advertising prompts.


Prepare credit card with cashback advertising.


In [7]:
ads = []

ads.append("Generate banking advertising mail of a credit card with 10% cashback.")

Prepare savings account advertising.


In [8]:
ads.append(
    "Generate banking advertising mail of a saving account with 8 percent interest."
)

Prepare educational course advertising.


In [9]:
ads.append("Generate advertising mail of an online JAVA programming course.")

Prepare photo editing software advertising.


In [10]:
ads.append(
    "Generate advertising mail of paid (10$ a month) photo editing software for professionals."
)

Prepare healthy food catering advertising.


In [11]:
ads.append("Generate advertising mail of healthy food catering with a free first meal.")

<a id="models"></a>

## Foundation Models on watsonx


#### List available chat models


In [12]:
models_json = httpx.get(
    endpoint_url + "/ml/v1/foundation_model_specs",
    headers={
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    },
    params={
        "limit": 50,
        "version": "2024-03-19",
        "filters": "function_text_chat,!lifecycle_withdrawn:and",
    },
).json()

models_ids = [m["model_id"] for m in models_json["resources"]]
models_ids

['ibm/granite-3-2-8b-instruct',
 'ibm/granite-3-3-8b-instruct',
 'ibm/granite-3-3-8b-instruct-np',
 'ibm/granite-3-8b-instruct',
 'ibm/granite-4-h-small',
 'ibm/granite-guardian-3-8b',
 'meta-llama/llama-3-2-11b-vision-instruct',
 'meta-llama/llama-3-2-90b-vision-instruct',
 'meta-llama/llama-3-3-70b-instruct',
 'meta-llama/llama-3-405b-instruct',
 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8',
 'meta-llama/llama-guard-3-11b-vision',
 'mistral-large-2512',
 'mistralai/mistral-medium-2505',
 'mistralai/mistral-small-3-1-24b-instruct-2503',
 'openai/gpt-oss-120b']

You need to specify `model_id` that will be used for inferencing:


In [13]:
model_id = "ibm/granite-3-3-8b-instruct"

<a id="predict"></a>

## Generate mail advertising


### Generate mail advertising using `watsonx.ai` model.

**Note:** You might need to adjust model parameters for different models or tasks, to do so please refer to <a href="https://cloud.ibm.com/apidocs/watsonx-ai#text-chat" target="_blank" rel="Infer text params">documentation</a>.


Initialize the `PromptClient` class.

**Hint:** Your authentication token might expire, if so please regenerate the `access_token` and reinitialize the `PromptClient` class.


In [14]:
prompt_client = PromptClient(access_token, project_id, endpoint_url)

Get the docs summaries.


In [15]:
results_low_temperature = []

for instruction in ads:
    results_low_temperature.append(
        prompt_client.chat(
            model_id=model_id,
            messages=[
                {"role": "user", "content": instruction},
            ],
            temperature=0,
        )
    )

Explore model outputs.


In [16]:
for i, (prompt, resp) in enumerate(zip(ads, results_low_temperature), start=1):
    content = resp["choices"][0]["message"]["content"]

    safe_content = content.replace("---", "—")

    display(
        Markdown(
            f"""
## 🧠 Prompt {i}
> {prompt}

#### ✉️ Generated Content
{safe_content}
"""
        )
    )


## 🧠 Prompt 1
> Generate banking advertising mail of a credit card with 10% cashback.

#### ✉️ Generated Content
📧 Subject: Unleash the Power of 10% Cashback with Our Premium Credit Card! 💳

Dear [Customer's Name],

We hope this message finds you well! We're thrilled to introduce our exclusive Premium Credit Card, designed to reward your everyday spending with an unbeatable 10% cashback on all purchases.

🌟 Here's what you can expect:

1. **10% Cashback on All Spend**: Earn a 10% cashback on every purchase you make using our Premium Credit Card. This means for every $100 you spend, you'll receive $10 back in your account.

2. **No Annual Fee**: Enjoy the benefits without any annual fees, making it an affordable way to maximize your savings.

3. **Secure Transactions**: Your security is our top priority. Our card comes with advanced fraud protection and chip technology for safe and secure transactions.

4. **Easy Cashback Redemption**: Redeem your cashback easily through our mobile app or online banking portal. You can choose to have the cashback credited to your account or used towards your next purchase.

5. **Additional Perks**: Enjoy additional benefits like complimentary travel insurance, extended warranty on products, and exclusive discounts at popular retailers.

Don't miss out on this opportunity to turn your everyday spending into significant savings. Apply for our Premium Credit Card today and start enjoying 10% cashback on all your purchases!

To apply, simply visit our website [insert link] or contact our customer service team at [insert contact details].

Thank you for choosing us for your banking needs. We look forward to serving you better with our Premium Credit Card.

Best Regards,

[Your Name]
[Your Position]
[Bank's Name]

📞 Contact Us: [Insert Contact Details] | Website: [Insert Bank's Website] | Email: [Insert Bank's Email]

*Please note: The 10% cashback is capped at a certain amount per month/quarter, as per the card's terms and conditions. For detailed terms, please refer to the card agreement available at the time of application.

🔒 Secure Your Finances: Apply Now for Our Premium Credit Card with 10% Cashback! 🔒



## 🧠 Prompt 2
> Generate banking advertising mail of a saving account with 8 percent interest.

#### ✉️ Generated Content
📧 Subject: Unlock Your Savings Potential with Our High-Yield Savings Account! 🔑

Dear [Customer's Name],

We hope this message finds you well! We're excited to introduce our new High-Yield Savings Account, designed to help you grow your savings with an impressive 8% annual interest rate.

🌟 Why choose our High-Yield Savings Account?

1. **Competitive Interest Rate**: Earn 8% annual interest on your savings, significantly higher than the national average.

2. **Low Minimum Balance**: No need to maintain a large balance to enjoy this attractive rate.

3. **FDIC Insured**: Your savings are protected by the Federal Deposit Insurance Corporation (FDIC) up to $250,000.

4. **Easy Access**: With online and mobile banking, managing your account is a breeze.

5. **No Monthly Maintenance Fees**: We believe in keeping your money where it belongs – in your account, not in fees.

🌟 How to Open an Account:

1. Visit our website or download our mobile app.
2. Click on 'Open an Account' and select 'High-Yield Savings Account'.
3. Follow the simple steps to provide your information and submit your application.

Don't miss out on this opportunity to maximize your savings. Open your High-Yield Savings Account today and let your money work harder for you!

If you have any questions or need assistance, please don't hesitate to contact our dedicated customer service team at [Support Email] or [Support Phone Number].

Thank you for choosing our bank for your financial needs. We look forward to helping you grow your savings!

Best regards,

[Your Name]
[Your Position]
[Bank's Name]

Disclaimer: The 8% annual interest rate is variable and subject to change based on market conditions. The rate displayed is accurate as of [Date]. Please refer to our website or contact our customer service for the most current rate.

*This is a fictional advertisement and does not represent a specific bank's current offerings. Always verify details with your bank before making financial decisions.*



## 🧠 Prompt 3
> Generate advertising mail of an online JAVA programming course.

#### ✉️ Generated Content
📣 Exciting News: Unlock Your Java Mastery with Our Online Course! 📚

Are you ready to elevate your programming skills and become a Java expert? Look no further! Our comprehensive online Java programming course is designed to take you from beginner to professional in no time.

🌟 What You'll Learn:
- Java fundamentals: syntax, data types, variables, and control structures
- Object-oriented programming concepts: classes, objects, inheritance, and polymorphism
- Advanced Java topics: exception handling, file I/O, and multithreading
- Java frameworks: Spring and Hibernate
- Real-world projects to solidify your skills

🎯 Why Choose Us:
- Expert instructors: Learn from experienced Java developers who have worked on large-scale projects
- Flexible learning: Study at your own pace, anytime, anywhere
- Hands-on approach: Gain practical experience with coding exercises and projects
- Lifetime access: Review lessons and materials whenever you want
- Certificate of completion: Showcase your new skills to potential employers

📅 Don't miss out on this opportunity to become a sought-after Java programmer! Enroll today and start your journey to success.

👉 Click here to enroll: [Insert Enrollment Link]

Join our community of successful Java developers and transform your career. See you in class! 🚀



## 🧠 Prompt 4
> Generate advertising mail of paid (10$ a month) photo editing software for professionals.

#### ✉️ Generated Content
📷 **Unleash Your Creativity with ProPhoto Master!** 📷

Are you a professional photographer or editor seeking the perfect tool to elevate your work? Look no further! Introducing ProPhoto Master – the ultimate photo editing software designed specifically for professionals like you.

**Why Choose ProPhoto Master?**

1. **Advanced Editing Tools**: Harness the power of professional-grade tools, including layer-based editing, customizable brushes, and intelligent selection refinement.

2. **Batch Processing**: Save time with our efficient batch processing feature, allowing you to apply edits to multiple images simultaneously.

3. **HDR Merge & Toning**: Create stunning HDR images with our intuitive merge and toning tools, ensuring perfect exposure and detail in every shot.

4. **Lens Correction & Perspective Control**: Correct lens distortions and adjust perspective with ease, ensuring your images are always perfectly aligned.

5. **Color Grading & Matching**: Achieve consistent color grading across your portfolio with our advanced color matching tools.

6. **Regular Updates & Support**: Stay ahead of the curve with regular updates and priority customer support, ensuring you always have access to the latest features and assistance.

**Special Offer: Just $10/month!**

For a limited time, we're offering ProPhoto Master at an unbeatable price of just $10 per month. Subscribe now and experience the difference in quality and efficiency that ProPhoto Master brings to your photo editing workflow.

**Upgrade Your Workflow Today!**

Don't miss out on this incredible opportunity to enhance your professional photo editing skills. Sign up for ProPhoto Master now and start creating breathtaking images that stand out from the crowd.

Visit our website or click the link below to start your 7-day free trial and experience the power of ProPhoto Master firsthand:

[Subscribe Now](www.prophotomaster.com/subscribe)

*ProPhoto Master – Empowering professionals to create, edit, and share their best work.*



## 🧠 Prompt 5
> Generate advertising mail of healthy food catering with a free first meal.

#### ✉️ Generated Content
📩 Subject: 🌱 Discover a Healthier You with Our Delicious Catering - Enjoy Your First Meal on Us! 🍽️

Dear [Customer's Name],

Are you looking to embrace a healthier lifestyle without compromising on taste? Look no further! We're thrilled to introduce our premium healthy food catering service, designed to nourish your body and delight your taste buds.

🌟 Why Choose Us?

1. **Nutritious & Delicious**: Our menu is packed with wholesome, organic ingredients, ensuring you get the best of both worlds - taste and nutrition.

2. **Customizable Options**: We cater to various dietary preferences and restrictions, including vegan, gluten-free, and paleo options.

3. **Expert Chefs**: Our team of experienced chefs crafts each meal with care, ensuring every bite is a culinary masterpiece.

4. **Convenient Delivery**: We understand your time is valuable. That's why we offer flexible delivery options to suit your schedule.

🎁 Special Offer for You:

To help you kickstart your health journey, we're excited to offer a FREE first meal with any of our catering plans! Simply choose your preferred plan, and we'll take care of the rest.

👉 Don't miss out on this opportunity to experience the difference our healthy catering can make. Visit our website [insert website link] or call us at [insert phone number] to place your order today.

Remember, every healthy choice you make is a step towards a happier, more vibrant you. Let us be a part of your journey.

Warm Regards,

[Your Name]
[Your Position]
[Company Name]

P.S. This offer is valid for a limited time only. Hurry and claim your free first meal today! 🍽️🌱


You might also try to change model parameters to see whether it provides better ads. In following cell we changed the model temperature to 1.0 to let the model select words more creatively.


In [17]:
results_high_temperature = []

for instruction in ads:
    results_high_temperature.append(
        prompt_client.chat(
            model_id=model_id,
            messages=[
                {"role": "user", "content": instruction},
            ],
            temperature=1,
        )
    )

Explore model outputs.


In [18]:
for i, (prompt, resp) in enumerate(zip(ads, results_high_temperature), start=1):
    content = resp["choices"][0]["message"]["content"]

    safe_content = content.replace("---", "—")

    display(
        Markdown(
            f"""
## 🧠 Prompt {i}
> {prompt}

#### ✉️ Generated Content
{safe_content}
"""
        )
    )


## 🧠 Prompt 1
> Generate banking advertising mail of a credit card with 10% cashback.

#### ✉️ Generated Content
📩 Subject: Unleash the Power of 10% Cashback with Our Premium Credit Card! 🔥

Dear [Customer's Name],

Are you tired of missing out on valuable rewards with your current credit card? 💳 It's time to upgrade to our Premium Credit Card, designed to offer you an unbeatable 10% cashback on all your eligible purchases! ⚡

With our card, you'll enjoy:

1. 🔥 10% Cashback on Eligible Purchases 🔥
   - Earn 10% cashback on your everyday spending at grocery stores, gas stations, and even dining outlets. The more you spend, the more you earn!

2. 🏆 Earn Rewards Faster 🤩
   - With no limits on the amount of cashback you can earn, watch your savings grow effortlessly. There's no cap, so the more you use your card, the more you'll reap the benefits!

3. 👑 Premium Benefits & Perks 👑
   - Access exclusive member-only discounts, special financing offers, and worldwide acceptance for seamless shopping experiences, both online and in-store.

4. 🔒 Secure & Convenient 🔒
   - Invest in safe transactions and fraud protection with our state-of-the-art security features, ensuring your peace of mind during every purchase.

Apply now and experience the ultimate in convenience, savings, and rewards! 🚀

[Call-to-Action Button: Apply for Your Premium Credit Card Today! 🏧]

Don't miss out on this fantastic opportunity to maximize your savings. Simply click the button below and complete your application in minutes!

👉 [Apply Now] 👈

*T&Cs apply. Cashback is credited monthly to your card statement. Eligible purchases are subject to terms and exclusions mentioned in the cardholder agreement.

Best regards,

[Your Name]
[Your Position]
[Bank Name]

P.S. Spread the word and share this offer with your friends and family. They'll thank you for introducing them to an exceptional credit card experience! 📝👍

—

Disclaimer: This generated text is an example of a banking advertising mail for a credit card offering 10% cashback. Make sure to customize it according to your bank's branding, policies, and legal requirements before using. Always abide by data privacy and security protocols.



## 🧠 Prompt 2
> Generate banking advertising mail of a saving account with 8 percent interest.

#### ✉️ Generated Content
💌 **Unlock Your Financial Future with Our High-Yield Savings Account!** 💌

Dear [Customer's Name],

Are you looking for a secure and rewarding way to grow your savings? Look no further! We're excited to introduce our High-Yield Savings Account, offering an attractive 8% annual interest rate on your deposits.

🌟 **Why Choose Our High-Yield Savings Account?**

1. **Competitive Interest Rate**: Earn 8% interest on your savings, significantly outpacing the national average. Your money works harder for you, growing faster than ever before.

2. **Safety and Security**: Your deposits are protected by FDIC insurance, giving you peace of mind and ensuring that your savings are always secure.

3. **Flexibility**: With no minimum balance requirements and low minimum opening deposit, you can easily manage your funds without worrying about penalty fees.

4. **Easy Access**: Access your money when you need it with free online and mobile banking, including transfers, bill payments, and more.

5. **Automatic Savings Plan**: Set up a recurring transfer from your checking to your savings account, making saving effortless and consistent.

🌟 **How to Open an Account**:

Opening an account is simple and quick. You can:

1. Visit our nearest branch.
2. Call our dedicated account specialists at [Phone Number].
3. Apply online through our secure website.

💡 **Start Saving for Your Dreams Today!** 💡

Don't let your hard-earned money sit idle. Open our High-Yield Savings Account and watch your savings grow with our competitive 8% interest rate. It's time to take control of your financial future.

Click here [Insert Link] to apply now or visit our website to learn more about our High-Yield Savings Account.

Thank you for choosing our bank as your trusted partner in building a prosperous future.

Best Regards,

[Your Name]
[Your Title]
[Bank Name]

*Please note: The 8% annual percentage yield (APY) is effective as of [Date] and may change at any time without prior notice. For detailed terms and conditions, visit our website or contact our customer service team.



## 🧠 Prompt 3
> Generate advertising mail of an online JAVA programming course.

#### ✉️ Generated Content
📢 **Unlock Your JAVA Mastery with Our Online Course!** 📢

🎯 **Why Choose Our JAVA Programming Course?**

1. **Expert Instructors**: Learn from experienced JAVA developers who have real-world experience.
2. **Comprehensive Curriculum**: From basics to advanced topics, we cover it all.
3. **Flexible Learning**: Study at your own pace, anytime, anywhere.
4. **Practical Projects**: Apply your skills with hands-on projects.
5. **Lifetime Access**: Revisit lessons and materials anytime you want.

💻 **What You'll Learn:**

- JAVA Fundamentals: Variables, Operators, Control Structures, Looping Constructs
- Object-Oriented Programming with JAVA: Classes, Objects, Inheritance, Polymorphism
- Exception Handling and Multithreading
- JAVA Development Tools: IDEs, Debugging, Version Control
- Building JAVA Applications: Desktop and Web Applications

🌟 **Who This Course Is For:**

- Beginners interested in learning JAVA programming.
- Developers looking to expand their skillset.
- Students preparing for JAVA-based interviews or certifications.

🔗 **Enroll Now and Start Your JAVA Journey!**

[Button: ENROLL NOW]

*Terms and Conditions Apply. Discount codes not applicable on bulk purchase.*

📸 **Student Testimonials:**

> "The course was excellent! The instructors were very knowledgeable and the practical projects helped solidify my understanding." - John D.
>
> "This course transformed my JAVA skills. The pace was perfect for someone coming from a non-programming background." - Sarah K.

🔐 **Secure Payment & Data Privacy:** We ensure all transactions are secure and your data is protected.

📅 **Limited Time Offer!** Enroll now and get a **20% discount** on our JAVA programming course. Use code JAVA20 at checkout.

*Hurry, offer valid only for the first 100 enrollments.*

📝 **Still have questions? Contact us at support@javaprogrammingcourse.com**



## 🧠 Prompt 4
> Generate advertising mail of paid (10$ a month) photo editing software for professionals.

#### ✉️ Generated Content
📩 Subject: elevate your photo editing game with ProEdit Pro - just $10/month! ⚡

Dear [Professional Photographer's Name],

Are you still relying on basic editing tools that restrict your creative potential? It's time to unlock your full artistic vision with ProEdit Pro – the ultimate photo editing software designed specifically for professionals like you!

🌟 Unleash unmatched precision and control:
- Advanced color grading & correction tools for flawless skin tones, vibrant colors, and stunning visuals.
- 360-degree editing capabilities for immersive panoramic images.
- Intelligent layer management for seamless compositing and masking.
- Professional-grade noise reduction and sharpening algorithms to bring out the intricate details in every shot.

🎯 Why settle for generic editing solutions when you deserve the best? With ProEdit Pro, you get:
- Exclusive access to a library of professional presets for various photo styles and genres, saving you hours in post-production.
- Seamless integration with popular camera brands and RAW file formats, ensuring the highest quality results.
- Cross-platform compatibility: Effortlessly work on Windows, macOS, and even Linux for ultimate flexibility.
- Regular updates and dedicated support for a smooth user experience.

🔒 Your investment in ProEdit Pro is not just a software purchase, but a commitment to your professional growth and elevated client satisfaction. For a limited time, unlock the full potential of your photography with an irresistible offer:

🔥 Just $10/month, with no long-term contracts! 🌟

Don't miss out on this opportunity to refine and elevate your photo editing process. Try ProEdit Pro risk-free with our 7-day money-back guarantee. Click here to get started today: [Insert Link]

We can't wait for you to experience the difference ProEdit Pro makes in your workflow!

Warm regards,

[Your Name]
[Your Position]
ProEdit Pro Team

P.S. - As a valued professional photographer, you'll receive a special 10% discount during signup. Use code PHOTOPRO10 at checkout.

—

This ad copy aims to present ProEdit Pro as a must-have, affordable solution tailored for professional photographers, highlighting its advanced features, seamless integration, and cross-platform accessibility. The enticing price point and the risk-free trial offer are key points to encourage immediate action.



## 🧠 Prompt 5
> Generate advertising mail of healthy food catering with a free first meal.

#### ✉️ Generated Content
📩 Subject: 🌿 Treat Your Taste Buds & Health with Our Free First Meal Offer! 🍽️

Dear [Customer's Name],

Are you constantly seeking delicious, nutritious meals that fuel your body and mind? Look no further! We're thrilled to introduce our premium healthy food catering service, designed to cater to your unique nutritional needs while satisfying your cravings.

🔥 **Exclusive Offer Just for You:** ☑️ We're excited to present our special promotion: A **FREE FIRST MEAL** on us!

🌱 **Why Choose Our Healthy Food Catering?**

1. **100% Natural & Organic Ingredients:** We source high-quality, fresh produce from trusted local farmers, ensuring you receive nothing but the best.

2. **Customizable Meal Plans:** Our professional dietitians work closely with you to create tailored meal plans that cater to your dietary preferences, restrictions, and goals (vegan, gluten-free, low-carb, etc.).

3. **Expertly Prepared Delicious Meals:** Our in-house chefs craft your meals with love and precision, balancing flavors and textures to create mouthwatering dishes that are as healthy as they are delightful.

4. **Convenient & Time-Saving:** Let us handle your meal prep, so you can focus on what truly matters. Our efficient delivery system ensures fresh meals arrive right at your doorstep.

🎁 **Claim Your Free First Meal Today:**

To celebrate our partnership and introduce you to our exceptional service, we're offering a COMPLImentary first meal! Simply follow this link to select your preferred meal and enjoy a scrumptious treat on the house: [Insert Link]

🌟 **Testimonials:**

"I've never felt better! Their customized meal plans have helped me reach my fitness goals while ensuring I don't compromise on taste." – Sarah M.

"Healthy food doesn't have to be boring. Their meals are so flavorful and satisfying. I can't imagine going back to my old eating habits." – John D.

📅 **Don't miss out on this limited-time offer!** Claim your free first meal today and embark on a journey towards optimal health and happiness. Contact us at support@healthyfoodcatering.com or call us at (123) 456-7890 to learn more.

We can't wait to nourish your body with our wholesome creations!

Warmest Regards,

[Your Name]
[Your Position]
[Healthy Food Catering Company Name]

*Please note: This promotion is valid for first-time customers only and cannot be combined with any other offers. Terms and conditions apply.

—

Don't hesitate – embrace a healthier lifestyle with our mouthwatering, nutrient-packed meals! Your body & taste buds will thank you. 🌿🍽️


You can also try changing the prompt and/or parameters to generate more detailed advertising. In the next cell, the photo editing software advertising prompt was transformed to provide more accurate results.


In [19]:
photo_software_prompt = "Write an advertising e-mail of paid (9$ a month, first month free) photo editing software COOLSOFTWARE targeted for early professionals."

In [20]:
results_photo_soft = prompt_client.chat(
    model_id=model_id,
    messages=[
        {"role": "user", "content": photo_software_prompt},
    ],
    temperature=1,
)

In [21]:
content = results_photo_soft["choices"][0]["message"]["content"]

safe_content = content.replace("---", "—")

display(
    Markdown(
        f"""
### ✉️ Generated Content
{safe_content}
"""
    )
)


### ✉️ Generated Content
📩 Subject: Unleash Your Creativity with COOLSOFTWARE - First Month FREE! 🌟

Hello Early Career Professionals,

Are you ready to elevate your photo editing skills and stand out in your field? Look no further than COOLSOFTWARE, the all-in-one photo editing solution designed to empower you with professional-grade tools.

🌟 Why choose COOLSOFTWARE? 🌟

1. **User-Friendly Interface**: COOLSOFTWARE boasts an intuitive design, making it easy for beginners to navigate while still offering advanced features for seasoned editors.

2. **Powerful Editing Tools**: From basic adjustments to complex layer management, our software provides an extensive range of tools to help you bring your vision to life.

3. **Time-Saving Shortcuts**: Streamline your workflow with customizable shortcuts and batch processing, ensuring you spend less time editing and more time creating.

4. **Seamless Integration**: COOLSOFTWARE works flawlessly with popular cameras, lenses, and other software, making it the perfect companion for your existing setup.

5. **Collaboration Features**: Share projects with team members, solicit feedback, and maintain version control – all within the COOLSOFTWARE ecosystem.

For a limited time, we're offering early career professionals like you a chance to experience the magic of COOLSOFTWARE with a FREE first month! After that, it's just $9 a month – an investment in your creative growth that'll pay off tenfold.

💻 Get Started Today:

1. Visit [coolsoftware.com/subscribe](http://coolsoftware.com/subscribe)
2. Choose your preferred subscription plan (just $9 a month, billed annually)
3. Enjoy your FREE first month!

Don't miss out on this opportunity to refine your skills and impress clients with stunning visuals. COOLSOFTWARE is your secret weapon for standing out in a competitive world.

🤝 We can't wait to see what you create!

Best regards,

[Your Name]
[Your Job Title]
COOLSOFTWARE Team

P.S. Remember, your first month is on us! No credit card required – simply sign up and start editing right away. 🚀


<a id="summary"></a>

## Summary and next steps

You successfully completed this notebook!

You learned how to generate mail advertising with LLM's on watsonx.

Check out our <a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="Online Documentation">Online Documentation</a> for more samples, tutorials, documentation, how-tos, and blog posts.


### Authors and Maintainers

**Kahila Mokhtari (Former)**, Senior Data Scientist at IBM watsonx.ai

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

**Karol Zmorski**, Software Engineer at IBM watsonx.ai

Copyright © 2026 IBM. This notebook and its source code are released under the terms of the MIT License.
